# Week 1 · Day 3 — Lab 2
## Reading & Writing Data: CSV, Parquet, JSON

> **AI Engineering Academy** · Gamut Technology Services · pandas 3.x

Every pipeline starts and ends with I/O. The three formats you'll meet constantly:
**CSV** (universal, human-readable, but untyped and slow), **Parquet** (columnar,
compressed, fully typed — the right choice for every pipeline intermediate), and
**JSON** (for nested/API payloads). The single most important habit this lab
builds: **declare dtypes at read time**, and **use Parquet, not CSV, for
intermediate storage** so types survive round-trips.

You'll ingest raw CSVs with explicit typing, round-trip through Parquet to prove
type fidelity, parse JSON (flat and nested), and see first-hand how CSV silently
destroys dtypes.

### Learning objectives
1. Read a CSV with `usecols`, `dtype`, `parse_dates`, and `na_values`, then inspect with `info()` / `isna()`.
2. Write and read **Parquet** with PyArrow, preserving dtypes and selecting columns.
3. Parse flat JSON with `read_json(orient="records")` and nested JSON with `json_normalize`.
4. Demonstrate why **CSV loses type information** and Parquet does not.

### Time budget — ~70 min
| Segment | Time |
|---|---|
| Framing & objectives | 5 min |
| **A.** Typed CSV ingestion | 15 min |
| **B.** Parquet round-trip | 16 min |
| **C.** JSON: flat & nested | 16 min |
| **D.** The CSV type-loss pitfall | 12 min |
| Wrap-up + stretch | 6 min |

### Files you need (in `data/`)
- `users.csv` — 2000 rows: `user_id, signup_date, plan, region, seats`.
- `events.csv` — 8000 rows: `event_id, user_id, event_date, model, score, input_tokens, output_tokens, latency_ms`.
- `events_sample.json` — 200 events, `orient="records"`.
- `sessions_nested.json` — 6 sessions, each with a nested `events` list.

*(This lab writes Parquet/JSON artifacts into a local `artifacts/` folder.)*


In [ ]:
import pandas as pd
import numpy as np
import json, os
from pathlib import Path

print("pandas", pd.__version__)   # target: pandas 3.x on Python 3.13

DATA = Path("data")

# Scratch dir for the Parquet/JSON artifacts you'll write in this lab.
ART = Path("artifacts")
ART.mkdir(exist_ok=True)

def check(label, predicate):
    try:
        ok = bool(predicate())
    except Exception as exc:
        ok = False
        label = f"{label}  (raised {type(exc).__name__}: {exc})"
    print(("PASS " if ok else "FAIL "), label)
    return ok

print("ready.")

## Part A — Typed CSV ingestion  *(guided)*

`read_csv` will *infer* types by scanning the file — slow and memory-hungry, and
it often guesses wrong (integers become floats when nulls are present). Declare
what you know: `usecols` to load only needed columns, `dtype` for types,
`parse_dates` for timestamps, `na_values` for sentinels.


In [ ]:
users = pd.read_csv(
    DATA / "users.csv",
    dtype={"user_id": "int32", "plan": "category", "region": "category"},
    parse_dates=["signup_date"],
)
print(users.dtypes)
print("rows:", len(users))

### Exercise A1 — Ingest events with explicit types
Read `events.csv` into `events` with: `user_id` as `int32`, `model` as
`category`, `score` as `float32`, and `event_date` parsed as a datetime. Then
capture the per-column null counts as `null_counts` (a Series from `isna().sum()`).


💡 **Hint.** Pass a `dtype={...}` dict plus `parse_dates=["event_date"]`. After
loading, `events.isna().sum()` gives a Series of null counts per column.


In [ ]:
events = None        # TODO: read_csv with dtype + parse_dates
null_counts = None   # TODO: events.isna().sum()

In [ ]:
check("A1: events has 8000 rows", lambda: len(events) == 8000)
check("A1: model is a category dtype",
      lambda: str(events["model"].dtype) == "category")
check("A1: score is float32", lambda: str(events["score"].dtype) == "float32")
check("A1: event_date parsed to datetime",
      lambda: str(events["event_date"].dtype).startswith("datetime64"))
check("A1: events has no nulls", lambda: int(null_counts.sum()) == 0)

## Part B — Parquet round-trip

Parquet is columnar, compressed, and carries an embedded schema, so dtypes survive
a write/read cycle exactly — and it's typically 3–5× smaller and faster than CSV.
It's the correct format for every pipeline intermediate.


In [ ]:
users.to_parquet(ART / "users.parquet", engine="pyarrow", compression="snappy")
back = pd.read_parquet(ART / "users.parquet", engine="pyarrow")
print("dtypes survived round-trip:")
print(back.dtypes)

### Exercise B1 — Write events to Parquet, then read back a subset
Write `events` to `artifacts/events.parquet` (PyArrow engine, snappy compression).
Then read **only** the `user_id` and `score` columns back into `events_slim` using
the `columns=` argument. Confirm `score` is still `float32` after the round-trip.


💡 **Hint.** `events.to_parquet(path, engine="pyarrow", compression="snappy")`,
then `pd.read_parquet(path, engine="pyarrow", columns=["user_id", "score"])`.
Parquet's columnar layout means only those two columns are read from disk.


In [ ]:
events_path = ART / "events.parquet"
# TODO: write events to events_path (pyarrow, snappy)

events_slim = None   # TODO: read back ONLY user_id and score

In [ ]:
check("B1: parquet file was written", lambda: (ART / "events.parquet").exists())
check("B1: only 2 columns read back",
      lambda: list(events_slim.columns) == ["user_id", "score"])
check("B1: score dtype survived as float32",
      lambda: str(events_slim["score"].dtype) == "float32")
check("B1: all 8000 rows present", lambda: len(events_slim) == 8000)

### Exercise B2 — Parquet is smaller than CSV
Write `events` to `artifacts/events_out.csv` (with `index=False`) and compare file
sizes. Set `csv_bytes` and `parquet_bytes` from `os.path.getsize(...)`, and
`parquet_smaller` to whether Parquet is the smaller file.


In [ ]:
csv_out = ART / "events_out.csv"
# TODO: write events to csv_out with index=False

csv_bytes = None         # TODO: os.path.getsize(csv_out)
parquet_bytes = None     # TODO: os.path.getsize(events_path)
parquet_smaller = None   # TODO: is parquet the smaller file?

In [ ]:
check("B2: both files exist", lambda: csv_out.exists() and events_path.exists())
check("B2: Parquet is smaller than CSV", lambda: parquet_smaller is True)

## Part C — JSON: flat and nested

`read_json(orient="records")` reads a flat list of `{col: val}` objects. For
**nested** payloads (the usual shape of an API response), `pd.json_normalize`
flattens a record path and pulls parent fields down via `meta`.


In [ ]:
# Flat: a list of record objects
flat = pd.read_json(DATA / "events_sample.json", orient="records")
print("flat shape:", flat.shape, "| columns:", list(flat.columns))
flat.head(3)

### Exercise C1 — Normalize a nested payload
`sessions_nested.json` looks like `{"data": [{session_id, user_id, events: [...]},
...]}`. Load it with the `json` module into `payload`, then use
`pd.json_normalize` with `record_path="events"` and `meta=["session_id",
"user_id"]` to flatten it into `norm` — one row per event, with the session fields
carried down.


💡 **Hint.** `payload = json.load(open(...))`, then
`pd.json_normalize(payload["data"], record_path="events", meta=["session_id",
"user_id"])`. Each of the 6 sessions has 10 events → 60 rows.


In [ ]:
with open(DATA / "sessions_nested.json") as f:
    payload = json.load(f)

norm = None      # TODO: json_normalize with record_path + meta

In [ ]:
check("C1: flat JSON read into a DataFrame",
      lambda: isinstance(flat, pd.DataFrame) and len(flat) == 200)
check("C1: normalized to one row per event (6x10)",
      lambda: len(norm) == 60)
check("C1: event fields present after flattening",
      lambda: {"event_id", "model", "score"} <= set(norm.columns))
check("C1: session meta carried down",
      lambda: {"session_id", "user_id"} <= set(norm.columns))

## Part D — The CSV type-loss pitfall

Here's why CSV is wrong for intermediates. Build a typed frame, round-trip it
through **CSV**, and watch the dtypes collapse. Then round-trip the same frame
through **Parquet** and watch them survive.


In [ ]:
typed = pd.DataFrame({
    "user_id": pd.array([1, 2, 3], dtype="int32"),
    "plan": pd.array(["free", "pro", "free"], dtype="category"),
    "signup": pd.to_datetime(["2024-01-01", "2024-02-01", "2024-03-01"]),
})
print("original dtypes:")
print(typed.dtypes)

### Exercise D1 — Prove CSV loses types, Parquet keeps them
Round-trip `typed` through CSV (`artifacts/typed.csv`, `index=False`) into
`from_csv`, and through Parquet (`artifacts/typed.parquet`) into `from_parquet`.
Then compare: set `csv_plan_dtype` and `parquet_plan_dtype` to the `str(...)` of
the `plan` column dtype from each.


💡 **Hint.** After a naive `to_csv` / `read_csv`, `plan` comes back as `str`
(the category is gone) and `signup` comes back as text. After `to_parquet` /
`read_parquet`, `plan` is still `category`.


In [ ]:
# CSV round-trip
typed.to_csv(ART / "typed.csv", index=False)
from_csv = None          # TODO: read the CSV back

# Parquet round-trip
typed.to_parquet(ART / "typed.parquet", engine="pyarrow")
from_parquet = None      # TODO: read the Parquet back

csv_plan_dtype = None        # TODO: str(from_csv["plan"].dtype)
parquet_plan_dtype = None    # TODO: str(from_parquet["plan"].dtype)

In [ ]:
check("D1: CSV lost the category dtype (now str)",
      lambda: csv_plan_dtype == "str")
check("D1: Parquet preserved the category dtype",
      lambda: parquet_plan_dtype == "category")
check("D1: CSV turned the datetime into plain text",
      lambda: str(from_csv["signup"].dtype) in ("str", "object"))

## Stretch goals *(for fast finishers)*

**S1 — Arrow-backed dtypes.** Re-read `artifacts/events.parquet` with
`dtype_backend="pyarrow"` into `events_arrow` and capture `str(events_arrow["score"].dtype)`
into `arrow_score_dtype` (it should mention `pyarrow`).

**S2 — JSON Lines output.** Write the first 100 rows of `events` to
`artifacts/events.jsonl` with `to_json(orient="records", lines=True)` and confirm
the file has exactly 100 lines (`line_count`).


In [ ]:
# S1
events_arrow = None      # TODO: read_parquet(..., dtype_backend="pyarrow")
arrow_score_dtype = None # TODO: str(events_arrow["score"].dtype)

# S2
jsonl_path = ART / "events.jsonl"
# TODO: write events.head(100) to jsonl_path (orient="records", lines=True)
line_count = None        # TODO: count lines in the file

In [ ]:
check("S1: score dtype is Arrow-backed",
      lambda: "pyarrow" in arrow_score_dtype)
check("S2: jsonl has exactly 100 lines", lambda: line_count == 100)

## Wrap-up — what you can now do

- Ingest CSV with explicit `dtype`, `usecols`, `parse_dates`, `na_values`, and inspect nulls immediately.
- Round-trip through **Parquet** with full dtype fidelity and column-selective reads.
- Parse flat JSON with `read_json` and nested JSON with `json_normalize`.
- Explain and demonstrate why CSV loses types and Parquet is the right intermediate format.

**Next:** Lab 3 — inspecting and selecting data with `head`/`info`/`describe`,
`.loc`/`.iloc`, boolean indexing, and `query()`.
